# Hardware and events

The code of [sparxml.dev/learn/hardware](https://sparxml.dev/learn/hardware/), run on a CPU runtime. The first cell installs sparx.

In [ ]:
%pip install -q 'sparxml[nir] @ git+https://github.com/AshishKumar4/sparx'

In [ ]:
import flax.linen as nn
import jax
import jax.numpy as jnp
import nir

from sparx.nir import from_nir, to_nir
from sparx.nn import LI, LIF

pilot = nn.Sequential([nn.Dense(64), LIF(tau=3.0, reset="zero"),
                       nn.Dense(64), LIF(tau=3.0, reset="zero"),
                       nn.Dense(2), LI(tau=5.0)])
variables = pilot.init(jax.random.key(0), jnp.zeros((1, 1, 7)))

# A step is 10 ms; NIR counts seconds.
graph = to_nir(pilot, variables, dt=0.01)
nir.write("pilot.nir", graph)

model, read = from_nir(nir.read("pilot.nir"), dt=0.01)
x = jax.random.normal(jax.random.key(1), (200, 8, 7))
difference = pilot.apply(variables, x) - model.apply(read, x)
print(jnp.abs(difference).max())                 # 0.0